### DLinear

1. 시계열 분해
2. 예측
3. 예측결과 합산

[출처] https://github.com/cure-lab/LTSF-Linear/blob/main/models/DLinear.py

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [8]:
# input=(batch, 시계열, 채널)
x = torch.randn(1, 10, 5)
print(x)
# 첫번째 시계열
print(x[:, 0:1, :])
# 첫번째 시계열 크기=(1, 1, 채널수)
print(x[:, 0:1, :].shape)
# 첫번째 시계열을 dim=1 축으로 3번 반복
print(x[:, 0:1, :].repeat(1, 3, 1))

# 마지막 시계열
print(x[:, -1:, :])
# 마지막 시계열 크기-(1, 1, 채널수)
print(x[:, -1:, :].shape)
# 마지막 시계열을 dim=1 축으로 3번 반복
print(x[:, -1:, :].repeat(1, 3, 1))

tensor([[[ 1.6850,  0.8438,  0.3754, -1.1625, -0.3086],
         [-0.4232, -0.0858,  0.9080,  0.3572, -0.8538],
         [ 0.5340,  0.4256, -0.0510,  0.2988, -0.1276],
         [-0.3417, -0.3644, -0.8870,  0.1109,  0.1587],
         [ 0.8362,  0.7622,  0.3570,  0.4917, -1.2017],
         [ 0.1644,  0.2065,  0.3208, -1.3880, -0.0188],
         [ 1.3153,  1.2600, -1.0122, -0.8618, -1.2477],
         [-0.4100,  0.9075, -2.2048, -0.8707,  0.7512],
         [ 0.3918, -1.7137,  0.2560,  2.9187,  0.1384],
         [-0.7608, -0.7524, -0.8396,  0.1432,  0.6169]]])
tensor([[[ 1.6850,  0.8438,  0.3754, -1.1625, -0.3086]]])
torch.Size([1, 1, 5])
tensor([[[ 1.6850,  0.8438,  0.3754, -1.1625, -0.3086],
         [ 1.6850,  0.8438,  0.3754, -1.1625, -0.3086],
         [ 1.6850,  0.8438,  0.3754, -1.1625, -0.3086]]])
tensor([[[-0.7608, -0.7524, -0.8396,  0.1432,  0.6169]]])
torch.Size([1, 1, 5])
tensor([[[-0.7608, -0.7524, -0.8396,  0.1432,  0.6169],
         [-0.7608, -0.7524, -0.8396,  0.1432,  0.616

In [9]:
print(x.transpose(1, 2).shape)
print(x.permute(0, 2, 1).shape)

torch.Size([1, 5, 10])
torch.Size([1, 5, 10])


In [10]:
# 1. 시계열 분해: trend(MA)와 seasonality(input-trend)
class moving_avg(nn.Module):
    def __init__(self, kernel_size, stride):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):
        # padding on the both ends of time series
        # 커널 사이즈 반만큼 앞뒤로 붙여주기 (pooling 후에 입력의 크기와 동일해짐)
        front = x[:, 0:1, :].repeat(1, (self.kernel_size - 1)//2, 1)
        end = x[:, -1:, :].repeat(1, (self.kernel_size-1)//2, 1)
        x = torch.cat([front, x, end], dim=1)
        # (batch, 시계열, 채널) -> (batch, 채널, 시계열)로 차원 변경 후 avg pooling
        x = self.avg(x.transpose(1,2))
        # 다시 원래대로 차원 변경
        x = x.permute(0, 2, 1)
        return x

class series_decomp(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.moving_avg = moving_avg(kernel_size, stride=1)

    def forward(self, x):
        trend = self.moving_avg(x)
        # seasonal: 원래 값에서 moving avg 뺀 값
        res = x - trend
        return res, trend

In [ ]:
# model
class Model(nn.Module):
    def __init__(self, configs):
        super().__init__()
        # lookback period
        self.seq_len = configs.seq_len
        # forecast period
        self.pred_len = configs.pred_len

        kernel_size = 25
        self.decomposition = series_decomp(kernel_size)
        self.individual = configs.individual
        self,channels = configs.channels

        if